# NVIDIA PersonaPlex — Google Colab Speech-to-Speech Test

This notebook runs **NVIDIA PersonaPlex locally in Google Colab** using the official `NVIDIA/personaplex` code and the gated `nvidia/personaplex-7b-v1` model weights.

Unlike Chatterbox Turbo, PersonaPlex is **not just text-to-speech**. It is a **full-duplex speech-to-speech conversational model**:

```text
your WAV / microphone
        ↓
   PersonaPlex
  ↙           ↘
agent text   agent speech
```

### What this notebook does

1. Checks the Colab GPU.
2. Installs the official NVIDIA PersonaPlex repository and dependencies.
3. Authenticates with Hugging Face.
4. Uploads a user speech WAV/MP3 file.
5. Converts it to **24 kHz mono WAV**.
6. Defines the PersonaPlex role/persona prompt.
7. Runs official **offline streaming evaluation**.
8. Plays the generated agent speech.
9. Displays the generated text output.
10. Lets you switch among NVIDIA's packaged voices.
11. Includes an optional live server section for experimenting with full-duplex interaction.

> **Before running:** open the model page `nvidia/personaplex-7b-v1` on Hugging Face and accept NVIDIA's model access/license conditions. A Hugging Face token with access to the model is required.


## 1. Runtime / GPU check

PersonaPlex is a large conversational speech model. NVIDIA documents A100 and H100 as supported hardware and reports its reference inference on an A100 80 GB.

If your Colab GPU has less VRAM, the official code supports `--cpu-offload`. This notebook uses that mode by default for the offline test.

In Colab select:

**Runtime → Change runtime type → GPU**


In [2]:
import torch
import subprocess

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    props = torch.cuda.get_device_properties(0)
    vram_gb = props.total_memory / 1024**3

    print("GPU:", gpu)
    print(f"VRAM: {vram_gb:.1f} GB")
else:
    print("WARNING: No CUDA GPU detected.")

!nvidia-smi


PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
VRAM: 14.6 GB
Tue Aug 11 06:59:10 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        | 

## 2. Install PersonaPlex

This follows NVIDIA's official repository setup:

- install the Opus development library;
- clone `NVIDIA/personaplex`;
- install the repository's `moshi` package;
- install `accelerate` so `--cpu-offload` is available.

After installation, a runtime restart may occasionally be required if Colab had incompatible packages preloaded.


In [3]:
!apt-get update -qq
!apt-get install -y -qq libopus-dev ffmpeg

%cd /content
!rm -rf personaplex
!git clone https://github.com/NVIDIA/personaplex.git
%cd /content/personaplex

!pip install -q ./moshi
!pip install -q accelerate huggingface_hub

print("PersonaPlex installation complete.")


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
/content
Cloning into 'personaplex'...
remote: Enumerating objects: 214, done.
remote: Counting objects: 100% (127/127), done.
remote: Compressing objects: 100% (80/80), done.
remote: Total 214 (delta 53), reused 47 (delta 47), pack-reused 87 (from 1)
Receiving objects: 100% (214/214), 1.44 MiB | 3.86 MiB/s, done.
Resolving deltas: 100% (55/55), done.
/content/personaplex
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 33.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 57.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.5/417.5 kB 28.8 MB/s 

## 3. Hugging Face authentication

The model repository is gated. You must:

1. Sign in to Hugging Face.
2. Accept access conditions for `nvidia/personaplex-7b-v1`.
3. Create a Hugging Face access token.

The cell below first looks for a Colab Secret named **`HF_TOKEN`**. If none exists, it asks for the token without printing it.


In [4]:
import os
from getpass import getpass

HF_TOKEN = None

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    pass

if not HF_TOKEN:
    HF_TOKEN = getpass("Enter your Hugging Face token: ").strip()

if not HF_TOKEN:
    raise ValueError("HF_TOKEN is required.")

os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN

print("HF token configured for this runtime.")


Enter your Hugging Face token: ··········
HF token configured for this runtime.


In [5]:
from huggingface_hub import hf_hub_download

# Small gated-file access check. This confirms that the token can access the repo.
try:
    config_path = hf_hub_download(
        repo_id="nvidia/personaplex-7b-v1",
        filename="config.json",
        token=HF_TOKEN,
    )
    print("Model access confirmed:", config_path)
except Exception as e:
    print("Could not access the model.")
    print("Make sure you accepted the PersonaPlex license on Hugging Face.")
    raise


config.json:   0%|          | 0.00/56.0 [00:00<?, ?B/s]

Model access confirmed: /root/.cache/huggingface/hub/models--nvidia--personaplex-7b-v1/snapshots/fdaf4090a61cb315c138a1faee287ffd6c716309/config.json


## 4. Choose the PersonaPlex voice

The official repository includes fixed voice-conditioning embeddings.

### Natural voices

```text
Female: NATF0, NATF1, NATF2, NATF3
Male:   NATM0, NATM1, NATM2, NATM3
```

### Variety voices

```text
Female: VARF0, VARF1, VARF2, VARF3, VARF4
Male:   VARM0, VARM1, VARM2, VARM3, VARM4
```

Start with `NATF2` or `NATM1` for a natural conversational test.


In [6]:
VOICE = "NATF2"

VOICE_FILE = f"{VOICE}.pt"

print("Selected voice:", VOICE_FILE)


Selected voice: NATF2.pt


## 5. Define the agent's role / persona

PersonaPlex uses a **text prompt** to tell the agent who it is and how it should behave.

For your conversational Bible-study style experiment, you can start with something like this:


In [7]:
TEXT_PROMPT = """
You enjoy having a thoughtful and natural conversation.
You are a calm, empathetic Christian Bible study companion.
Listen carefully to the user before responding.
Speak conversationally rather than sounding like a lecture.
When relevant, explain Biblical ideas clearly and mention scripture naturally.
Do not overwhelm the user with too many verses at once.
Allow pauses, interruptions, and changes of thought naturally.
""".strip()

PROMPT_PATH = "/content/personaplex_prompt.txt"

with open(PROMPT_PATH, "w", encoding="utf-8") as f:
    f.write(TEXT_PROMPT)

print(TEXT_PROMPT)


You enjoy having a thoughtful and natural conversation.
You are a calm, empathetic Christian Bible study companion.
Listen carefully to the user before responding.
Speak conversationally rather than sounding like a lecture.
When relevant, explain Biblical ideas clearly and mention scripture naturally.
Do not overwhelm the user with too many verses at once.
Allow pauses, interruptions, and changes of thought naturally.


## 6. Upload user speech

PersonaPlex consumes user audio rather than plain text.

Upload a short recording of yourself asking the model something, for example:

> “I've been struggling with uncertainty lately. How should I think about faith when I don't know what will happen next?”

WAV is ideal, but this notebook accepts common audio formats and converts them with `ffmpeg`.


In [13]:
from google.colab import files
from pathlib import Path

uploaded = files.upload()

if not uploaded:
    raise RuntimeError("No audio file uploaded.")

uploaded_name = next(iter(uploaded))
INPUT_ORIGINAL = f"/content/{uploaded_name}"

print("Uploaded:", INPUT_ORIGINAL)


Saving indian female.mp3 to indian female (1).mp3
Uploaded: /content/indian female (1).mp3


## 7. Convert input audio to PersonaPlex format

PersonaPlex expects **24 kHz audio**. We also normalize the test input to mono PCM WAV so the offline runner receives a predictable format.


In [8]:
from google.colab import files
import os
import subprocess
from IPython.display import Audio, display

uploaded = files.upload()

# Automatically get uploaded filename
INPUT_ORIGINAL = next(iter(uploaded.keys()))

print("Uploaded file:", INPUT_ORIGINAL)

INPUT_WAV = "/content/personaplex_input_24k_mono.wav"

subprocess.run(
    [
        "ffmpeg",
        "-y",
        "-i", INPUT_ORIGINAL,
        "-ar", "24000",
        "-ac", "1",
        "-c:a", "pcm_s16le",
        INPUT_WAV
    ],
    check=True
)

print("Prepared:", INPUT_WAV)

display(Audio(INPUT_WAV))


Saving indian female.mp3 to indian female.mp3
Uploaded file: indian female.mp3
Prepared: /content/personaplex_input_24k_mono.wav


# 8. Run PersonaPlex offline

NVIDIA's official offline runner **streams the input WAV through the model** and captures the model's output stream into a WAV file.

This is different from Chatterbox's `model.generate(text)`: PersonaPlex is processing incoming speech and producing conversational speech.

`--cpu-offload` is enabled below because Colab GPU memory varies. If you have an A100 80 GB and want maximum speed, set `CPU_OFFLOAD = False`.


In [11]:
import os
import subprocess
import shlex
import time

OUTPUT_WAV = "/content/personaplex_output.wav"
OUTPUT_TEXT = "/content/personaplex_output.json"

CPU_OFFLOAD = True
SEED = 42424242

cmd = [
    "python", "-m", "moshi.offline",
    "--voice-prompt", VOICE_FILE,
    "--text-prompt", TEXT_PROMPT,
    "--input-wav", INPUT_WAV,
    "--seed", str(SEED),
    "--output-wav", OUTPUT_WAV,
    "--output-text", OUTPUT_TEXT,
]

if CPU_OFFLOAD:
    cmd.append("--cpu-offload")

print("Running:")
print(" ".join(shlex.quote(x) for x in cmd))

start = time.perf_counter()

env = os.environ.copy()
env["HF_TOKEN"] = HF_TOKEN



result = subprocess.run(
    cmd,
    cwd="/content/personaplex",
    env=env,
    capture_output=True,
    text=True
)

elapsed = time.perf_counter() - start

print("========== STDOUT ==========")
print(result.stdout)

print("\n========== STDERR ==========")
print(result.stderr)

print("\nReturn code:", result.returncode)

if result.returncode != 0:
    raise RuntimeError(
        f"PersonaPlex exited with code {result.returncode}\n\n"
        f"{result.stderr}"
    )

print(f"Finished in {elapsed:.2f} seconds")
print("Audio:", OUTPUT_WAV)
print("Text:", OUTPUT_TEXT)


Running:
python -m moshi.offline --voice-prompt NATF2.pt --text-prompt 'You enjoy having a thoughtful and natural conversation.
You are a calm, empathetic Christian Bible study companion.
Listen carefully to the user before responding.
Speak conversationally rather than sounding like a lecture.
When relevant, explain Biblical ideas clearly and mention scripture naturally.
Do not overwhelm the user with too many verses at once.
Allow pauses, interruptions, and changes of thought naturally.' --input-wav /content/personaplex_input_24k_mono.wav --seed 42424242 --output-wav /content/personaplex_output.wav --output-text /content/personaplex_output.json
========== STDOUT ==========
[Info] retrieving voice prompts
[Info] voice_prompt_dir = /root/.cache/huggingface/hub/models--nvidia--personaplex-7b-v1/snapshots/fdaf4090a61cb315c138a1faee287ffd6c716309/voices
[Info] loading mimi
[Info] mimi loaded
[Info] loading moshi


========== STDERR ==========
Traceback (most recent call last):
  File "<fr

RuntimeError: PersonaPlex exited with code 1

Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/local/lib/python3.12/dist-packages/moshi/offline.py", line 431, in <module>
    main()
  File "/usr/local/lib/python3.12/dist-packages/moshi/offline.py", line 408, in main
    run_inference(
  File "/usr/local/lib/python3.12/dist-packages/moshi/offline.py", line 206, in run_inference
    lm = loaders.get_moshi_lm(moshi_weight, device=device, cpu_offload=cpu_offload)
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/moshi/models/loaders.py", line 213, in get_moshi_lm
    state_dict = load_file(filename, device=dev.type)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/safetensors/torch.py", line 315, in load_file
    result[k] = f.get_tensor(k)
              

## 9. Play PersonaPlex output


In [10]:
from IPython.display import Audio, display
import os

if not os.path.exists(OUTPUT_WAV):
    raise FileNotFoundError(OUTPUT_WAV)

display(Audio(OUTPUT_WAV))


FileNotFoundError: /content/personaplex_output.wav

## 10. Inspect generated text

PersonaPlex can output text alongside the generated audio. The exact JSON structure can evolve, so this cell prints the returned file without assuming a fixed schema.


In [ ]:
import json
from pprint import pprint

if os.path.exists(OUTPUT_TEXT):
    with open(OUTPUT_TEXT, "r", encoding="utf-8") as f:
        data = json.load(f)

    pprint(data)
else:
    print("No output-text file found.")


# 11. Test another voice

You do **not** need to reinstall or rewrite the notebook. Change `VOICE`, then rerun the offline inference cell.

Example:


In [ ]:
VOICE = "NATM1"
VOICE_FILE = f"{VOICE}.pt"

print("Next run will use:", VOICE_FILE)


# 12. Assistant-role baseline test

NVIDIA documents this prompt as the baseline QA-assistant role:

```text
You are a wise and friendly teacher. Answer questions or provide advice in a clear and engaging way.
```

Use it when you want to evaluate PersonaPlex itself before testing a more specialized persona.


In [ ]:
TEXT_PROMPT = (
    "You are a wise and friendly teacher. "
    "Answer questions or provide advice in a clear and engaging way."
)

print(TEXT_PROMPT)


# 13. OPTIONAL — launch the real-time PersonaPlex server

The offline test above is the easiest reliable test inside a notebook.

PersonaPlex also ships with a **live full-duplex Web UI/server**. The official server listens on port `8998`.

Colab does not expose that local port directly to your browser, so the following cells start the PersonaPlex server and create a temporary Cloudflare tunnel. Browser microphone/WebSocket behavior through notebook tunnels can vary; for serious latency testing, run the same server on a GPU machine with a normal public HTTPS endpoint.


In [ ]:
# Download cloudflared for an optional public tunnel.
!wget -q -O /content/cloudflared     https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /content/cloudflared

print("cloudflared installed.")


In [ ]:
import os
import subprocess
import tempfile
import time

# Stop old test processes if this cell is rerun.
subprocess.run(["pkill", "-f", "moshi.server"], check=False)
subprocess.run(["pkill", "-f", "cloudflared"], check=False)

SSL_DIR = tempfile.mkdtemp(prefix="personaplex_ssl_")

server_cmd = [
    "python", "-m", "moshi.server",
    "--ssl", SSL_DIR,
    "--cpu-offload",
]

server_log = open("/content/personaplex_server.log", "w")

server_process = subprocess.Popen(
    server_cmd,
    cwd="/content/personaplex",
    env=os.environ.copy(),
    stdout=server_log,
    stderr=subprocess.STDOUT,
)

print("PersonaPlex server PID:", server_process.pid)
print("Waiting for server startup...")

time.sleep(15)

print(open("/content/personaplex_server.log", "r", errors="ignore").read()[-4000:])


In [ ]:
import subprocess
import time
import re

tunnel_log_path = "/content/cloudflared.log"
tunnel_log = open(tunnel_log_path, "w")

tunnel_process = subprocess.Popen(
    [
        "/content/cloudflared",
        "tunnel",
        "--url", "https://localhost:8998",
        "--no-tls-verify",
    ],
    stdout=tunnel_log,
    stderr=subprocess.STDOUT,
)

print("Cloudflare tunnel PID:", tunnel_process.pid)

public_url = None

for _ in range(30):
    time.sleep(1)

    try:
        log_text = open(tunnel_log_path, "r", errors="ignore").read()
    except Exception:
        continue

    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", log_text)

    if match:
        public_url = match.group(0)
        break

if public_url:
    print("\nOpen this URL in a new browser tab:")
    print(public_url)
else:
    print("Tunnel URL was not detected yet.")
    print(open(tunnel_log_path, "r", errors="ignore").read()[-4000:])


# What to evaluate

When comparing PersonaPlex against your current Chatterbox experiment, listen for:

- **Time to first spoken response**
- **Turn-taking**
- **Whether the model handles a pause naturally**
- **Whether you can interrupt it**
- **Whether it stops / adapts during a barge-in**
- **Voice naturalness**
- **Role/persona consistency**
- **Response intelligence**
- **GPU memory use**
- **Real-time stability over a longer conversation**

The important architectural difference is:

```text
Chatterbox Turbo test
text → TTS → audio

PersonaPlex test
live/user audio ⇄ conversational speech model ⇄ agent audio
```

That makes PersonaPlex directly relevant when testing a real speech-to-speech conversational application rather than only a TTS component.
